# Cohort 01: trial/block summaries and problem-2 trial cap

Run from the top. Summaries cover problems 1–4 and retain individual-mouse traces with a single all-mice distribution, without sub-01–04/sub-05–08 groups. Problem 2 uses a fixed 510-trial cap for pooled true-value rank proportions and total reversals. Mice with fewer trials retain all available trials and are flagged. Task names containing `mag` or `pretraining` are excluded, matching the cohort-03 loading rule.


In [ ]:
import importlib
import sys
from pathlib import Path
from unittest.mock import patch
from IPython.display import display

repo_root = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
                  if (p / "src" / "behavior_import").is_dir()), None)
if repo_root is None:
    raise RuntimeError("Open this notebook from within the magnitude-bandit-analysis repository.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
import_data_module = importlib.import_module("src.behavior_import.import_data")
# Refresh extraction when using a kernel opened before the tower-change fix.
importlib.reload(importlib.import_module("src.behavior_import.extract_trials"))
task = "open-field"
cohort = "cohort-01"
folder_name = "3x3_field_magnitude_bandit"
root = f"/Volumes/behrens/meg/{folder_name}/rawdata/{cohort}/"
if not Path(root).is_dir():
    raise FileNotFoundError(f"Mount the data drive; raw data folder not found: {root}")
excluded_task_name_parts = ("mag", "pretraining")
summary_problem_numbers = {1, 2, 3, 4}
fixed_trial_budget = 510
show_trial_block_summary_plots = True


In [ ]:
def task_name_from_df(df):
    required_columns = {"type", "subtype", "content"}
    if not required_columns.issubset(df.columns):
        return None

    task_name_rows = df.loc[
        df["type"].eq("info") & df["subtype"].eq("task_name"),
        "content",
    ]
    if task_name_rows.empty:
        return None
    return str(task_name_rows.iloc[0])


all_tsv_files = import_data_module.collect_tsv_files(Path(root).resolve())
eligible_tsv_files = []
for tsv_path, df in all_tsv_files:
    task_name = task_name_from_df(df)
    if task_name is None:
        print(f"[WARN] Skipping {tsv_path}: no task name found.")
        continue
    if any(part in task_name.casefold() for part in excluded_task_name_parts):
        print(f"[INFO] Skipping {tsv_path}: excluded task name {task_name!r}.")
        continue
    eligible_tsv_files.append((tsv_path, df))

print(f"[INFO] Keeping {len(eligible_tsv_files)} of {len(all_tsv_files)} TSV file(s).")
with patch.object(import_data_module, "collect_tsv_files", return_value=eligible_tsv_files):
    subjects_data = import_data_module.import_data(root)

if not subjects_data:
    raise ValueError("No sessions remain after filtering task names.")


## Trial and block summaries

Blocks continue across sessions; observed edge blocks may be incomplete. Time plots use daily means. Problem averages give each mouse equal weight.


In [ ]:
import importlib
summary_analysis = importlib.reload(importlib.import_module("src.behavior_analysis.trial_block_summary"))
summary_plotting = importlib.reload(importlib.import_module("src.behavior_visualization.plot_trial_block_summary"))
from src.behavior_import.extract_trials import extract_trials_grouped_by_problem

summary_problems = extract_trials_grouped_by_problem(subjects_data)
if summary_problem_numbers is not None:
    missing = set(summary_problem_numbers) - set(summary_problems)
    if missing:
        raise ValueError(f"Requested summary problems not found: {sorted(missing)}")
    summary_problems = {p: d for p, d in summary_problems.items() if p in summary_problem_numbers}
if not summary_problems:
    raise ValueError("Select at least one available problem for the summaries.")
subject_numbers = summary_analysis.subject_numbers_from_paths(path for path, _ in eligible_tsv_files)
session_trial_table, block_trial_table = summary_analysis.build_trial_block_tables(summary_problems, subject_numbers)
selection_label = "all-problems" if summary_problem_numbers is None else "problems-" + "-".join(map(str, sorted(summary_problem_numbers)))
summary_save_dir = repo_root / "results" / "figures" / task / cohort / "trial-block-summaries" / selection_label
summary_plot_paths = summary_plotting.plot_trial_block_summaries(
    session_trial_table, block_trial_table, summary_save_dir,
    show=show_trial_block_summary_plots, split_subject_groups=False)
print(f"Saved six summary figures (PNG/PDF) and source tables to {summary_save_dir}")


## Problem 2: first 510 trials

Pooled-trial true-value rank proportions (with the existing block-quality filters) and total reversals including confirmed terminal events. Saved separately from the full-data summaries.


In [ ]:
# Independent of problem_numbers: this comparison always uses problem 2.
import importlib
import json
import pandas as pd
from src.behavior_import.extract_trials import extract_trials_grouped_by_problem
from src.behavior_analysis.trial_block_summary import subject_numbers_from_paths
from src.behavior_analysis.get_good_reversal_info import get_good_reversal_info
from src.behavior_analysis.get_rank_counts_by_good_reversal import get_rank_counts_by_good_reversal
from src.behavior_analysis.get_diagnostic_p_value import pvalue_paired_t_best_vs_second_vs_third
from src.behavior_analysis.get_total_reversals import get_total_reversals
from src.behavior_visualization.plot_rank_proportions import plot_rank_proportions
from src.behavior_visualization.plot_num_reversals import plot_num_reversals
budget_module = importlib.reload(importlib.import_module("src.behavior_analysis.matched_trial_budget"))

budget_problems = extract_trials_grouped_by_problem(subjects_data)
if 2 not in budget_problems:
    raise ValueError("Problem 2 is not available.")
budget_subject_numbers = subject_numbers_from_paths(path for path, _ in eligible_tsv_files)
problem2_matched_trials, problem2_budget_report = budget_module.match_trial_budget(
    budget_problems[2], budget_subject_numbers, trial_budget=fixed_trial_budget)
X = problem2_budget_report["trial_budget"]
print(f"Problem 2: first {X} trials per mouse (fixed budget; shorter records are flagged).")
display(pd.DataFrame(problem2_budget_report["mice"]))

matched_save_dir = repo_root / "results" / "figures" / task / cohort / "problem-02" / f"first-{X}-trials"
matched_save_dir.mkdir(parents=True, exist_ok=True)
(matched_save_dir / "trial_budget.json").write_text(json.dumps(problem2_budget_report, indent=2) + "\n")
problem2_matched_windows = get_good_reversal_info(problem2_matched_trials, include_first_block=True)
problem2_matched_ranks = get_rank_counts_by_good_reversal(problem2_matched_windows)
problem2_matched_pvalues = pvalue_paired_t_best_vs_second_vs_third(
    problem2_matched_ranks, pool_trials_within_subject=True)
plot_rank_proportions(problem2_matched_ranks,
    average_across_mice_pvalues=problem2_matched_pvalues,
    pool_trials_within_subject=True,
    save_path=str(matched_save_dir / f"Rank Proportions (True Value) (Pool Trials True) (First {X} Trials)"))
plot_num_reversals(problem2_matched_trials,
    save_path=matched_save_dir / f"Total Reversals (First {X} Trials)")
problem2_matched_totals = {mouse: get_total_reversals(sessions) for mouse, sessions in problem2_matched_trials.items()}
pd.DataFrame.from_dict(problem2_matched_totals, orient="index").to_csv(matched_save_dir / "total_reversals.csv", index_label="mouse")
(matched_save_dir / "rank_counts_and_statistics.json").write_text(json.dumps(
    {"counts": problem2_matched_ranks, "p_values": problem2_matched_pvalues}, indent=2, default=str) + "\n")
print(f"Saved problem-2 matched-trial analysis to {matched_save_dir}")


## Reversals over completed trials: problems 1–3

Full-data cumulative reversals and trailing 100-trial reversal rates (per 100 trials), both across mice and by mouse. Includes confirmed terminal reversals. Early windows use their observed exposure; across-mouse means include only mice that reached each trial count. Run after setup and data loading.


In [ ]:
reversal_problem_numbers = {1, 2, 3}
reversal_trial_window = 100

import importlib
reversal_plotting = importlib.reload(importlib.import_module("src.behavior_visualization.plot_reversals_by_trial"))
from src.behavior_import.extract_trials import extract_trials_grouped_by_problem

# Full problem data, independent of the 510-trial comparison above.
reversal_problems = extract_trials_grouped_by_problem(subjects_data)
missing = reversal_problem_numbers - set(reversal_problems)
if missing:
    raise ValueError(f"Requested reversal problems not found: {sorted(missing)}")
for problem_number in sorted(reversal_problem_numbers):
    subjects_trials = reversal_problems[problem_number]
    save_dir = repo_root / "results" / "figures" / task / cohort / f"problem-{problem_number:02d}" / "reversal-stats"
    reversal_plotting.plot_reversals_by_trial(
        subjects_trials, threshold=10,
        save_path=save_dir / "Cumulative Reversals Over Time (By Trial)")
    reversal_plotting.plot_reversals_by_trial(
        subjects_trials, window=reversal_trial_window,
        save_path=save_dir / f"Moving Average Reversals Over Time (By Trial, {reversal_trial_window}-Trial Window)")
    print(f"Saved trial-based reversal plots for problem {problem_number} to {save_dir}")
